# SAFE MoE — Mixture-of-Experts por Cluster (apenas `y_heads`)

Pipeline completo:
1. Geração de dataset (mesmo do baseline, com `cluster_*` features)
2. Normalização e split temporal
3. Treinamento do **VehicleForecastingSafeMoEModel** (base global + correções por cluster)
4. Avaliação com métricas de manutenção
5. Exportação ONNX
6. **Comparação Baseline vs SAFE MoE**

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

In [ ]:
from api.config.dataset_config import DatasetConfig
from api.config.training_config import TrainingConfig
from api.config.model_config import ModelConfig
from api.config.output_config import OutputConfig

from moviasai.data.utils import load_raw_data
from moviasai.forecasting.training_pipeline import MoETrainingPipeline

## Pipeline SAFE MoE

Função `run_moe_training(target)` — idêntica à `run_training` mas usa `MoETrainingPipeline`.

In [ ]:
def run_moe_training(target: str) -> dict:
    """Executa o pipeline SAFE MoE para o target especificado ('km' ou 'h')."""
    dataset_cfg = DatasetConfig.from_yaml('../config/dataset_config.yaml')
    training_cfg = TrainingConfig.from_yaml('../config/training_config.yaml')
    model_cfg = ModelConfig.from_yaml('../config/model_config.yaml')
    output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')

    df_daily = load_raw_data(output_cfg.train_data_path(target), target=target)

    print(f'Target:     {target}')
    print(f'Epochs:     {training_cfg.trainer.max_epochs}')
    print(f'Batch size: {training_cfg.data.batch_size}')
    print(f'Modelo:     SAFE MoE')

    pipeline = MoETrainingPipeline.from_config(
        df_daily=df_daily.to_pandas(),
        dataset_cfg=dataset_cfg,
        training_cfg=training_cfg,
        model_cfg=model_cfg,
        output_config=output_cfg,
        target=target,
    )

    metrics = pipeline.run()
    for split_name in ('val', 'test'):
        if split_name not in metrics:
            continue
        maint = metrics[split_name]['maintenance']
        print(f'\n=== {split_name.upper()} ===')
        for k_label in ('k2', 'k3', 'k4'):
            m = maint[k_label]
            k = int(k_label[1])
            print(f"\n  Limite {k} semanas:")
            print(f"    Erro médio:          {m['mean_error']:+.2f} dias")
            print(f"    Erro absoluto médio: {m['mae_days']:.2f} dias")
            print(f"    P90 erro:            {m['p90_error']:+.2f} dias")
            print(f"    % atrasos:           {m['pct_late']:.1f}%")
            print(f"    % adiantamentos:     {m['pct_early']:.1f}%")
    return metrics

## Treinar SAFE MoE

In [ ]:
moe_metrics_km = run_moe_training('km')
moe_metrics_h = run_moe_training('h')

Target:     km
Epochs:     10
Batch size: 64
Modelo:     SAFE MoE
✓ Modelo PKL carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST.pkl
  Nome: stage2_km
  Features: 5
  Classes: 3

✓ Carregado: 6718 veículos, 229760 versões
  - Extractors: 4
    • segmentation_features_km: 5 features
    • weekday_features_km: 49 features
    • month_phase_features_km: 15 features
    • monthly_cycle_features_km: 7 features
  - Classificador: stage2_km

GERANDO DATASET - KM
Versões disponíveis: 229760
Veículos únicos: 6718
Critérios de qualidade:
  • min_weeks_general: 12
  • num_weeks_recent: 4
  • min_recent_active_days: 5
Horizonte:
  • head_weeks: [1, 1, 1, 1] (total=4 semanas)
  • n_heads: 4
  • head_days: [7, 7, 7, 7]



100%|██████████| 229760/229760 [01:09<00:00, 3298.86it/s]



✓ Geração concluída
  • Amostras aceitas: 115,803
  • Taxa de aceitação: 50.4%

Rejeições por critério:
  • min_weeks: 80,144 (70.3%)
  • no_recent_data: 11,115 (9.8%)
  • min_active_days: 2,199 (1.9%)
  • no_future_data: 20,499 (18.0%)

✓ Dataset salvo em: C:\Users\f0pi\git\apimovias\data\.dataset_cache\km
  • Amostras: 115,803
  • Features: 78
  • Heads: 4


Seed set to 13


train_test_split temporal:
  Cutoff:       2025-08-17
  Treino:       100,600 amostras (86.9%) | 2025-03-23 → 2025-08-17 (22 semanas)
  Teste:        15,203 amostras (13.1%) | 2025-08-24 → 2025-09-07 (3 semanas)
train_test_split temporal:
  Cutoff:       2025-07-20
  Treino:       80,695 amostras (80.2%) | 2025-03-23 → 2025-07-20 (18 semanas)
  Teste:        19,905 amostras (19.8%) | 2025-07-27 → 2025-08-17 (4 semanas)


...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.



  MÉTRICAS — VAL

  HEADS (agregado semanal):
    head_0 (7d):  MAE=7.67  RMSE=11.41
    head_1 (7d):  MAE=8.80  RMSE=13.14
    head_2 (7d):  MAE=9.59  RMSE=14.26
    head_3 (7d):  MAE=10.27  RMSE=15.21

  DAILY (dias 1–7):
    MAE médio:  1.53
    RMSE médio: 2.66

  DECISÃO DE MANUTENÇÃO:

  Limite: 2 semanas (k=2)
    Erro médio:          +1.30 dias
    Erro absoluto médio: 1.87 dias
    P90 erro:            +6.48 dias
    % atrasos:           29.1%
    % adiantamentos:     8.3%

  Limite: 3 semanas (k=3)
    Erro médio:          +0.13 dias
    Erro absoluto médio: 0.19 dias
    P90 erro:            +0.00 dias
    % atrasos:           5.0%
    % adiantamentos:     0.4%

  Limite: 4 semanas (k=4)
    Erro médio:          -0.01 dias
    Erro absoluto médio: 0.01 dias
    P90 erro:            +0.00 dias
    % atrasos:           0.2%
    % adiantamentos:     0.1%


  MÉTRICAS — TEST

  HEADS (agregado semanal):
    head_0 (7d):  MAE=7.70  RMSE=11.81
    head_1 (7d):  MAE=8.86  RMSE=13.

W0424 10:38:24.114000 79704 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0424 10:38:24.116000 79704 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0424 10:38:24.117000 79704 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `VehicleForecastingSafeMoEModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VehicleForecastingSafeMoEModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Relatório gerado: C:\Users\f0pi\git\apimovias\logs\training\h\training_report_h.pdf

=== VAL ===

  Limite 2 semanas:
    Erro médio:          +1.30 dias
    Erro absoluto médio: 1.87 dias
    P90 erro:            +6.48 dias
    % atrasos:           29.1%
    % adiantamentos:     8.3%

  Limite 3 semanas:
    Erro médio:          +0.13 dias
    Erro absoluto médio: 0.19 dias
    P90 erro:            +0.00 dias
    % atrasos:           5.0%
    % adiantamentos:     0.4%

  Limite 4 semanas:
    Err

## Comparação Baseline vs SAFE MoE

Treinar baseline com a mesma configuração para comparação directa.

In [ ]:
def run_baseline(target: str) -> dict:
    """Executa o pipeline baseline para comparação."""
    from moviasai.forecasting.training_pipeline import MultiHeadTrainingPipeline

    dataset_cfg = DatasetConfig.from_yaml('../config/dataset_config.yaml')
    training_cfg = TrainingConfig.from_yaml('../config/training_config.yaml')
    model_cfg = ModelConfig.from_yaml('../config/model_config.yaml')
    output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')

    df_daily = load_raw_data(output_cfg.train_data_path(target), target=target)

    pipeline = MultiHeadTrainingPipeline.from_config(
        df_daily=df_daily.to_pandas(),
        dataset_cfg=dataset_cfg,
        training_cfg=training_cfg,
        model_cfg=model_cfg,
        output_config=output_cfg,
        target=target,
    )
    return pipeline.run()

# Descomentar para treinar baseline fresco (ou usar métricas do training.ipynb)
baseline_km = run_baseline('km')
baseline_h = run_baseline('h')

### Tabela comparativa — Métricas de manutenção

In [ ]:
import pandas as pd

def compare_metrics(baseline_metrics: dict, moe_metrics: dict, split: str = 'val') -> pd.DataFrame:
    """Compara métricas de manutenção entre baseline e SAFE MoE."""
    rows = []
    for k_label in ('k2', 'k3', 'k4'):
        base = baseline_metrics[split]['maintenance'][k_label]
        moe = moe_metrics[split]['maintenance'][k_label]
        k = int(k_label[1])
        for metric in ('mean_error', 'mae_days', 'p90_error', 'pct_late', 'pct_early'):
            rows.append({
                'k': k,
                'metric': metric,
                'baseline': base[metric],
                'moe': moe[metric],
                'delta': moe[metric] - base[metric],
            })
    df = pd.DataFrame(rows)
    df['melhor'] = df.apply(
        lambda r: '✅' if (r['metric'] == 'pct_early' and r['delta'] > 0) or
                          (r['metric'] != 'pct_early' and r['delta'] < 0)
                  else ('⚠️' if r['delta'] == 0 else '❌'),
        axis=1,
    )
    return df

# Exemplo (descomentar quando tiver ambos os resultados):
# compare_metrics(baseline_km, moe_metrics_km, 'val')

### P90_k2 e % atrasos por cluster

Análise per-cluster: verifica se o MoE melhora nos clusters críticos sem degradar os restantes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compare_per_cluster(baseline_metrics: dict, moe_metrics: dict,
                        split: str = 'val', k: int = 2):
    """Bar chart comparando P90 e % atrasos por cluster para k semanas."""
    import re
    cluster_re = re.compile(r'^cluster_\d+$')

    for label, metrics in [('Baseline', baseline_metrics), ('SAFE MoE', moe_metrics)]:
        meta = metrics[split]['metadata']
        errors = metrics[split]['maintenance'][f'k{k}']['errors']
        # Cluster = argmax das colunas cluster_*
        cluster_cols = sorted([c for c in meta.columns if cluster_re.match(c)])
        if not cluster_cols:
            # Usar coluna 'cluster' do metadata se disponível
            clusters = meta.get('cluster', np.zeros(len(meta)))
        else:
            clusters = meta[cluster_cols].values.argmax(axis=1)

    # Computar métricas por cluster para ambos
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric_name, metric_fn in [
        (axes[0], f'P90 erro (k={k})', lambda e: np.nanpercentile(e, 90)),
        (axes[1], f'% atrasos (k={k})', lambda e: float(np.nanmean(e > 0) * 100)),
    ]:
        for model_label, model_metrics in [('Baseline', baseline_metrics), ('MoE', moe_metrics)]:
            meta = model_metrics[split]['metadata']
            errors = model_metrics[split]['maintenance'][f'k{k}']['errors']
            cluster_col = meta.get('cluster', pd.Series(np.zeros(len(meta))))
            unique_clusters = sorted(cluster_col.unique())
            vals = [metric_fn(errors[cluster_col == c]) for c in unique_clusters]
            x = np.arange(len(unique_clusters))
            width = 0.35
            offset = -width/2 if model_label == 'Baseline' else width/2
            ax.bar(x + offset, vals, width, label=model_label)

        ax.set_xlabel('Cluster')
        ax.set_ylabel(metric_name)
        ax.set_title(metric_name)
        ax.set_xticks(x)
        ax.set_xticklabels([str(c) for c in unique_clusters])
        ax.legend()

    plt.tight_layout()
    plt.show()

# Exemplo:
# compare_per_cluster(baseline_km, moe_metrics_km, 'val', k=2)

### Critérios de sucesso

| Critério | Condição |
|---|---|
| ✅ P90_k2 global | Reduzido vs baseline |
| ✅ P90_k2 clusters críticos | Reduzido nos clusters com maior P90 |
| ✅ k=3 estável | P90_k3 não piora |
| ✅ y_daily intacto | MAE daily não degrada |
| ✅ Estabilidade | Variância entre runs controlada |